# Notebook 1 / 8 — Non-Classical Logics in Multi-Agent Communication: Basics

> **Series.** This is the first of eight notebooks exploring how non-classical logics help agents communicate.
> 1. **Basics** *(this notebook)* — eight foundational logics, one short scenario each
> 2. *Advanced* — fourteen rarer logics with richer expressive power
> 3. *Synthesis* — cross-logic benchmarks and research-style conclusions
> 4. *Applications* — ten real-world domains where each logic earns its keep
> 5. *Language* — non-classical logics applied to natural-language tasks
> 6. *Workflow* — an end-to-end pipeline composing the best of the above
> 7. *LangGraph* — the same pipeline rebuilt as a LangGraph state machine
> 8. *Experimental composition* — probing what happens when two non-classical logics overlap on the same linguistic phenomenon

## What this notebook is

A sketchbook exploring how different non-classical logics shape the way agents exchange, update, and reconcile information. Each section gives a 2-line intuition, a minimal hand-rolled evaluator, a small communication scenario, and a note on what classical logic would have lost.

No external logic libraries — everything is hand-rolled so the mechanics are visible.

## The logics in this notebook — definitions and references

### 1. Łukasiewicz Ł3 (three-valued logic)
**Definition.** A propositional logic with truth values `{0, ½, 1}`. Conjunction is `min`, disjunction `max`, negation `1 − x`, implication `min(1, 1 − x + y)`. The middle value `½` is *designated* as a third semantic status, not as "either true or false".
**Reference.** Łukasiewicz, J. (1920). *O logice trójwartościowej*. Ruch Filozoficzny 5, 170–171. English in Łukasiewicz, *Selected Works* (1970).

### 2. Fuzzy logic
**Definition.** Truth values form the real interval `[0, 1]`. Connectives are *t-norms* (for ∧) and *t-conorms* (for ∨). Common choices: `min/max` (Gödel), `x·y / x+y−xy` (product), `max(0,x+y−1) / min(1,x+y)` (Łukasiewicz). Models graded membership and confidence.
**Reference.** Zadeh, L. A. (1965). *Fuzzy sets*. Information and Control 8(3), 338–353.

### 3. Modal logic K / S4 / S5
**Definition.** Adds operators `□` ("necessarily") and `◇` ("possibly") interpreted over Kripke frames `(W, R, V)`. `□φ` holds at `w` iff `φ` holds at every `w′` with `w R w′`. K is the minimal normal modal logic; S4 adds reflexivity + transitivity (axioms T and 4); S5 adds symmetry, making `R` an equivalence.
**Reference.** Kripke, S. A. (1963). *Semantical analysis of modal logic I: normal modal propositional calculi*. Zeitschrift für mathematische Logik und Grundlagen der Mathematik 9, 67–96. Textbook: Blackburn, de Rijke & Venema, *Modal Logic* (CUP 2001).

### 4. Epistemic logic
**Definition.** A multi-agent modal logic where each agent `i` has its own accessibility relation `R_i`; the operator `K_i φ` reads "agent `i` knows that `φ`". Public announcements restrict the model to worlds where the announced formula holds.
**Reference.** Hintikka, J. (1962). *Knowledge and Belief*. Cornell University Press. Modern survey: Fagin, Halpern, Moses & Vardi, *Reasoning about Knowledge* (MIT 1995).

### 5. Paraconsistent LP (Logic of Paradox)
**Definition.** Three-valued logic with values `{F}`, `{T}`, `{T,F}`. A formula is *designated* (treated as a "true assertion") iff `T` is in its value. Crucially, `p ∧ ¬p` is satisfiable, so the rule `p, ¬p ⊢ q` (ex falso) fails — the system is paraconsistent.
**Reference.** Priest, G. (1979). *The logic of paradox*. Journal of Philosophical Logic 8(1), 219–241.

### 6. Intuitionistic logic
**Definition.** A constructive logic in which `φ` holds at an information state `s` only if a constructive witness exists. Negation `¬φ` means "no future state ever forces `φ`". Excluded middle `φ ∨ ¬φ` is *not* a tautology. Kripke gave a sound and complete semantics in terms of partially ordered information states.
**Reference.** Heyting, A. (1930). *Die formalen Regeln der intuitionistischen Logik*. Sitzungsberichte der Preußischen Akademie. Kripke semantics: Kripke, S. A. (1965). *Semantical analysis of intuitionistic logic I*.

### 7. Relevance logic / FDE (First Degree Entailment)
**Definition.** Four-valued logic with values `{}`, `{T}`, `{F}`, `{T,F}`. Entailment requires that the conclusion's truth value depends on the premises' content — irrelevant inferences like `p ⊨ q ∨ ¬q` are blocked.
**Reference.** Anderson, A. R. & Belnap, N. D. (1975). *Entailment: The Logic of Relevance and Necessity, Vol. 1*. Princeton University Press. Belnap, N. D. (1977). *A useful four-valued logic*.

### 8. Linear Temporal Logic (LTL)
**Definition.** A propositional temporal logic interpreted over linear traces (sequences of states). Operators: `X φ` (next), `F φ` (eventually), `G φ` (always), `φ U ψ` (until). Used for protocol specification and model checking.
**Reference.** Pnueli, A. (1977). *The temporal logic of programs*. 18th Annual Symposium on Foundations of Computer Science (FOCS), 46–57.

### 9. Default logic (Reiter)
**Definition.** A non-monotonic logic. A *default* `α : β / γ` is read "if `α` is known and `β` is consistent with the current beliefs, conclude `γ`". An *extension* of a knowledge base is a fixed point under such rules. Adding new facts can shrink an extension — i.e. retract conclusions.
**Reference.** Reiter, R. (1980). *A logic for default reasoning*. Artificial Intelligence 13(1–2), 81–132.

## Quick-glance table

| § | Logic | One-line intuition |
|---|---|---|
| 1 | Łukasiewicz Ł3 | Three truth values; *unknown* survives. |
| 2 | Fuzzy | Continuous `[0,1]` truth, t-norms/t-conorms. |
| 3 | Modal K/S4/S5 | `□`/`◇` over Kripke frames. |
| 4 | Epistemic | Per-agent `K_i` with public announcements. |
| 5 | Paraconsistent LP | Contradictions contained, no explosion. |
| 6 | Intuitionistic | Truth requires a constructive witness. |
| 7 | Relevance / FDE | Four values, blocks irrelevant inference. |
| 8 | LTL | `X`, `F`, `G`, `U` over linear traces. |
| 9 | Default logic | Tentative conclusions, retractable. |

Each section is self-contained: data structures are not reused across sections.

In [ ]:
from dataclasses import dataclass, field
from itertools import product
from typing import Callable, Dict, FrozenSet, Iterable, List, Set, Tuple
import math

def banner(title):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

## 1. Łukasiewicz 3-valued logic (Ł3)

Truth values: `0` (false), `½` (unknown/indeterminate), `1` (true).
Useful when an agent genuinely *doesn't know* and refuses to collapse that into false.

**Scenario.** Two sensor agents report on whether a door is open. Agent A is confident it's open; agent B's sensor is occluded and returns `½`. We combine their reports with Ł3 conjunction and disjunction and see what a controller agent concludes.

In [ ]:
F, U, T = 0.0, 0.5, 1.0

def l3_not(a):   return 1 - a
def l3_and(a,b): return min(a,b)
def l3_or(a,b):  return max(a,b)
def l3_imp(a,b): return min(1, 1 - a + b)  # Łukasiewicz implication

reports = {"AgentA": T, "AgentB": U}
print("Reports:", reports)

joint_and = l3_and(reports["AgentA"], reports["AgentB"])
joint_or  = l3_or (reports["AgentA"], reports["AgentB"])
print(f"Controller: both-say-open = {joint_and}  (½ = still uncertain)")
print(f"Controller: either-says-open = {joint_or}  (1 = act on it)")
print("\nClassical logic would have forced AgentB's ½ to either 0 or 1,")
print("losing the distinction between 'no' and 'cannot tell'.")

## 2. Fuzzy logic (product t-norm)

Continuous `[0,1]` truth, useful for graded belief / confidence fusion.

**Scenario.** Three scout agents report how strongly they believe an enemy is in zone Z. The commander fuses them with a product t-norm (joint evidence) and a probabilistic sum (at-least-one-sees-it).

In [ ]:
def fuzzy_and(a,b): return a*b                  # product t-norm
def fuzzy_or(a,b):  return a + b - a*b          # probabilistic sum
def fuzzy_not(a):   return 1 - a

scouts = {"S1": 0.9, "S2": 0.6, "S3": 0.2}
from functools import reduce
joint     = reduce(fuzzy_and, scouts.values())
any_sees  = reduce(fuzzy_or,  scouts.values())
print(f"All three confident: {joint:.3f}")
print(f"At least one confident: {any_sees:.3f}")
print("\nA classical vote would have thrown away the magnitude of disagreement.")

## 3. Modal logic K / S4 / S5 (Kripke semantics)

A Kripke model is `(W, R, V)`: worlds, accessibility, valuation. `□φ` holds at `w` iff `φ` holds at every `w'` with `w R w'`.

- **K**: no constraints on R
- **S4**: R reflexive + transitive
- **S5**: R equivalence relation

**Scenario.** One agent's accessibility relation encodes *what it considers possible*. We test whether `□p → p` (axiom T) holds depending on reflexivity.

In [ ]:
@dataclass
class Kripke:
    worlds: Set[str]
    R: Set[Tuple[str, str]]
    V: Dict[str, Set[str]]   # prop -> worlds where it is true

    def holds(self, formula, w):
        op = formula[0]
        if op == "atom": return w in self.V.get(formula[1], set())
        if op == "not":  return not self.holds(formula[1], w)
        if op == "and":  return self.holds(formula[1], w) and self.holds(formula[2], w)
        if op == "or":   return self.holds(formula[1], w) or  self.holds(fo mula[2], w)
        if op == "imp":  return (not self.holds(formula[1], w)) or self.hol s(formula[2], w)
        if op == "box":  return all(self.holds(formula[1], v) for (u,v) in self.R if u == w)
        if op == "dia":  return any(self.holds(formula[1], v) for (u,v) in self.R if u == w)
        raise ValueError(op)

# Non-reflexive K model
m_K = Kripke({"w1","w2"}, {("w1","w2")}, {"p": {"w2"}})
# S4/S5 style — add reflexive edge at w1
m_S4 = Kripke({"w1","w2"}, {("w1","w1"),("w1","w2"),("w2","w2")}, {"p": {"w2"}})

phi = ("imp", ("box", ("atom","p")), ("atom","p"))
print("K  model, □p → p at w1:", m_K.holds(phi, "w1"),  "(axiom T fails)")
print("S4 model, □p → p at w1:", m_S4.holds(phi, "w1"), "(axiom T holds)")

## 4. Epistemic logic — multi-agent knowledge

Each agent `i` has its own accessibility relation `R_i`. `K_i φ` = "agent i knows φ".

**Scenario: the Muddy Children (2 children).** Each child can see the other but not their own forehead. A parent announces "at least one of you is muddy." We track what each child knows before and after the announcement.

In [ ]:
@dataclass
class EpistemicModel:
    worlds: Set[str]
    R: Dict[str, Set[Tuple[str,str]]]   # agent -> relation
    V: Dict[str, Set[str]]

    def knows(self, agent, formula, w):
        return all(self._holds(formula, v) for (u,v) in self.R[agent] if u == w)

    def _holds(self, f, w):
        if f[0]=="atom": return w in self.V.get(f[1], set())
        if f[0]=="not":  return not self._holds(f[1], w)
        if f[0]=="and":  return self._holds(f[1], w) and self._holds(f[2], w)
        if f[0]=="or":   return self._holds(f[1], w) or  self._holds(f[2], w)
        if f[0]=="K":    return self.knows(f[1], f[2], w)
        raise ValueError(f)

    def public_announce(self, formula):
        """Restrict to worlds where formula holds; drop inaccessible pairs."""
        kept = {w for w in self.worlds if self._holds(formula, w)}
        R2 = {a: {(u,v) for (u,v) in rel if u in kept and v in kept}
              for a, rel in self.R.items()}
        V2 = {p: s & kept for p, s in self.V.items()}
        return EpistemicModel(kept, R2, V2)

# Worlds labelled by (m1, m2) where 1 = muddy.
worlds = {"00","01","10","11"}
# Child 1 can't tell its own state -> worlds differing only in 1st bit are linked
R1 = {(a,b) for a in worlds for b in worlds if a[1]==b[1]}
R2 = {(a,b) for a in worlds for b in worlds if a[0]==b[0]}
V  = {"m1": {w for w in worlds if w[0]=="1"},
      "m2": {w for w in worlds if w[1]=="1"}}
M = EpistemicModel(worlds, {"c1": R1, "c2": R2}, V)

actual = "11"  # both muddy
print("Before announcement:")
print("  c1 knows m1?", M.knows("c1", ("atom","m1"), actual))
print("  c2 knows m2?", M.knows("c2", ("atom","m2"), actual))

# Parent: "at least one of you is muddy"
at_least_one = ("or", ("atom","m1"), ("atom","m2"))
M1 = M.public_announce(at_least_one)
print("\nAfter 1st announcement (at least one muddy):")
print("  c1 knows m1?", M1.knows("c1", ("atom","m1"), actual))

# Second announcement: "neither of you yet knows your own state"
neither_knows = ("and",
                 ("not", ("K","c1",("atom","m1"))),
                 ("not", ("K","c2",("atom","m2"))))
M2 = M1.public_announce(neither_knows)
print("\nAfter 2nd announcement (neither knew):")
print("  c1 knows m1?", M2.knows("c1", ("atom","m1"), actual))
print("  c2 knows m2?", M2.knows("c2", ("atom","m2"), actual))
print("\nClassical one-shot reasoning can't model 'learning from another agent's ignorance'.")

## 5. Paraconsistent logic — LP (Logic of Paradox)

Truth values: `{F}`, `{T}`, `{T,F}` (both). Contradictions are *contained*, not explosive.

**Scenario.** Two agents exchange reports about a target's state. A says `p`, B says `¬p`. A classical knowledge base would explode (`p ∧ ¬p ⊢ q` for any q). LP lets the controller keep reasoning about *other* propositions unaffected.

In [ ]:
# Values as frozensets over {"T","F"}
Tv = frozenset({"T"}); Fv = frozenset({"F"}); Bv = frozenset({"T","F"})

def lp_not(v): return frozenset({"T" if x=="F" else "F" for x in v})
def lp_and(a,b):
    out = set()
    if "T" in a and "T" in b: out.add("T")
    if "F" in a or  "F" in b: out.add("F")
    return frozenset(out)
def lp_or(a,b):
    out = set()
    if "T" in a or  "T" in b: out.add("T")
    if "F" in a and "F" in b: out.add("F")
    return frozenset(out)
def designated(v): return "T" in v   # LP consequence: preserve truth

# Agent reports
kb = {"p": Tv, "q": Tv}
# B sends ¬p
def merge(v1, v2):
    return frozenset(v1 | v2)   # union = treat both as possible
kb["p"] = merge(kb["p"], Fv)
print("After merging contradictory reports on p:", kb)
print("Is q still considered true?", designated(kb["q"]))
print("Can we still use p ∨ q?", designated(lp_or(kb["p"], kb["q"])))
print("\nClassically, p ∧ ¬p would entail anything. LP isolates the clash to p alone.")

## 6. Intuitionistic logic (Kripke forcing)

Truth grows along a partial order of *information states*. `¬p` at state `s` means: at no future state is `p` established. Naturally models agents that only assert what they can constructively justify.

**Scenario.** An agent progressively receives evidence. We watch `p ∨ ¬p` — the law of excluded middle — which should *not* be forced until enough is known.

In [ ]:
@dataclass
class IntModel:
    states: List[str]
    leq: Set[Tuple[str,str]]          # reflexive, transitive order
    V: Dict[str, Set[str]]            # monotone: if p at s and s ≤ t then p at t

    def future(self, s):
        return [t for (u,t) in self.leq if u == s]

    def forces(self, f, s):
        if f[0]=="atom": return s in self.V.get(f[1], set())
        if f[0]=="and":  return self.forces(f[1],s) and self.forces(f[2],s)
        if f[0]=="or":   return self.forces(f[1],s) or  self.forces(f[2],s)
        if f[0]=="imp":  return all((not self.forces(f[1],t)) or self.forces(f[2],t) for t in self.future(s))
        if f[0]=="not":  return all(not self.forces(f[1],t) for t in self.future(s))
        raise ValueError(f)

states = ["s0","s1"]
leq    = {("s0","s0"),("s1","s1"),("s0","s1")}
V      = {"p": {"s1"}}   # p only known later
M = IntModel(states, leq, V)
lem = ("or", ("atom","p"), ("not", ("atom","p")))
print("p ∨ ¬p at s0 (uninformed):", M.forces(lem, "s0"))
print("p ∨ ¬p at s1 (p observed):", M.forces(lem, "s1"))
print("\nAn intuitionistic agent refuses to assert p ∨ ¬p until it has grounds.")

## 7. Relevance logic — FDE (First Degree Entailment)

Four values: `{}` (neither), `{T}`, `{F}`, `{T,F}`. Blocks irrelevant inference like `p ⊢ q ∨ ¬q`. Good when agents must justify *why* a conclusion follows from what was actually communicated.

In [ ]:
N = frozenset()            # neither
def fde_not(v):
    return frozenset({"T" if x=="F" else "F" for x in v})
def fde_and(a,b): return lp_and(a,b)   # same truth/false conditions
def fde_or(a,b):  return lp_or(a,b)

# Entailment: in every assignment, if premises are at least T then conclusion is at least T
VALUES = [N, Tv, Fv, Bv]
def fde_entails(premises, conclusion, atoms):
    for assignment in product(VALUES, repeat=len(atoms)):
        env = dict(zip(atoms, assignment))
        def ev(f):
            if f[0]=="atom": return env[f[1]]
            if f[0]=="not":  return fde_not(ev(f[1]))
            if f[0]=="and":  return fde_and(ev(f[1]), ev(f[2]))
            if f[0]=="or":   return fde_or (ev(f[1]), ev(f[2]))
        if all("T" in ev(p) for p in premises) and "T" not in ev(conclusion):
            return False
    return True

p = ("atom","p"); q = ("atom","q")
lem_q = ("or", q, ("not", q))
print("FDE: p ⊨ q ∨ ¬q ?", fde_entails([p], lem_q, ["p","q"]))
print("FDE: p, p→q ⊨ q ? (modus ponens needs material implication, skipped)")
print("FDE: p ∧ q ⊨ p ?", fde_entails([("and",p,q)], p, ["p","q"]))
print("\nRelevance blocks 'irrelevant' inferences that classical logic accepts.")

## 8. Linear Temporal Logic (LTL) over traces

Operators: `X` (next), `G` (globally), `F` (eventually), `U` (until). We evaluate over a finite trace produced by a simulated agent exchange.

In [ ]:
def ltl_eval(f, trace, i=0):
    n = len(trace)
    if i >= n: return False
    op = f[0]
    if op=="atom": return f[1] in trace[i]
    if op=="not":  return not ltl_eval(f[1], trace, i)
    if op=="and":  return ltl_eval(f[1], trace, i) and ltl_eval(f[2], trace, i)
    if op=="or":   return ltl_eval(f[1], trace, i) or  ltl_eval(f[2], trace, i)
    if op=="imp":  return (not ltl_eval(f[1], trace, i)) or ltl_eval(f[2], trace, i)
    if op=="X":    return i+1 < n and ltl_eval(f[1], trace, i+1)
    if op=="G":    return all(ltl_eval(f[1], trace, j) for j in range(i,n))
    if op=="F":    return any(ltl_eval(f[1], trace, j) for j in range(i,n))
    if op=="U":
        for j in range(i,n):
            if ltl_eval(f[2], trace, j): return True
            if not ltl_eval(f[1], trace, j): return False
        return False
    raise ValueError(op)

# Two agents negotiating; trace records atoms true at each step
trace = [
    {"request"},
    {"request","ack"},
    {"ack","propose"},
    {"propose","accept"},
    {"accept","done"},
]
for step, s in enumerate(trace):
    print(step, s)

req_leads_to_done = ("G", ("imp", ("atom","request"), ("F", ("atom","done"))))
print("\nG(request → F done):", ltl_eval(req_leads_to_done, trace))
print("F accept:", ltl_eval(("F",("atom","accept")), trace))
print("request U accept:", ltl_eval(("U",("atom","request"),("atom","accept")), trace))

## 9. Non-monotonic — default logic (Reiter-style, toy)

A default `α : β / γ` reads: *if α is known and β is consistent with what we know, conclude γ*. New information can retract γ.

**Scenario.** Agent A tells the controller "Tweety is a bird." Default: birds fly unless known otherwise. Agent B later adds "Tweety is a penguin." Classical logic can't gracefully retract; default logic does.

In [ ]:
@dataclass
class Default:
    pre: str
    justification: str
    conclusion: str

def extend(facts: Set[str], defaults: List[Default]) -> Set[str]:
    out = set(facts)
    changed = True
    while changed:
        changed = False
        for d in defaults:
            if d.pre in out and ("not_"+d.justification) not in out and d.conclusion not in out:
                out.add(d.conclusion); changed = True
    return out

defaults = [Default("bird", "flies", "flies")]

facts = {"bird"}
print("After A's report:", extend(facts, defaults))

facts = {"bird", "penguin", "not_flies"}  # penguin blocks the justification
print("After B's follow-up:", extend(facts, defaults))
print("\nDefault logic lets the controller tentatively conclude and later retract,")
print("which is exactly what inter-agent updates often demand.")

## Summary & next steps

| Logic | Best when agents need to... |
|---|---|
| Ł3 / fuzzy | express graded or unknown confidence |
| Modal K/S4/S5 | reason about possibility and necessity |
| Epistemic | model *each other's* knowledge, reason about announcements |
| LP (paraconsistent) | tolerate contradictory reports without exploding |
| Intuitionistic | only assert what they can justify constructively |
| FDE (relevance) | require inferences to actually use what was said |
| LTL | reason about protocol traces over time |
| Default logic | tentatively conclude and retract on new info |

**Natural next experiments:**
- Combine *epistemic* + *paraconsistent*: agents who know contradictory things without exploding.
- Combine *temporal* + *epistemic*: evolution of common knowledge over a protocol trace.
- Benchmark: same negotiation scenario solved under each logic, scored on agreement quality and robustness to noisy reports.